# Stock Analytics Dashboard

This notebook provides comprehensive stock analytics dashboards with interactive visualizations,
statistics, and benchmarking organized by **Phase 9.3 Feature Categories**.

**Version:** 2.3.0 | **Model Version:** v9_11 | **Updated:** 2026-01-12

## Dashboard Structure (21 Feature Categories, 460+ features)

### 📊 Core Analytics
1. **Region & Sector Overview** - Distribution analysis with feature coverage heatmaps
2. **Exchange Analysis** - Market-level analytics by stock exchange
3. **Industry Deep-Dive** - Granular industry-level benchmarking

### 💹 Financial Analytics (by Feature Category)
4. **Valuation Ratios** - P/E, P/B, EV/EBITDA, PEG analysis
5. **Profitability** - ROE, ROA, ROIC, margin analysis
6. **Growth Metrics** - Revenue, earnings, EBITDA growth, 5Y CAGR (NEW v1.15)
7. **Leverage & Liquidity** - Debt ratios, coverage, liquidity metrics

### 📈 Technical & Market Analytics
8. **Momentum & Technical** - Price momentum, EMA crossovers, breakout signals
9. **Technical Analysis** - RSI, 52-week range, volume momentum
10. **Market Sentiment** - Beta stability, systematic risk

### 🎯 Quality & Risk Analytics
11. **Quality & Risk** - Altman Z, accounting quality, distress signals
12. **Composite Scores** - Piotroski F, Beneish M, momentum scores
13. **Earnings Quality** - EPS surprises, GAAP vs Adjusted, EPS trajectory (NEW v1.15)

### 💰 Dividends & Capital Allocation
14. **Dividend Reliability** - Dividend timing, streaks, safety scores (ENHANCED v1.15)
15. **Capital Allocation** - CapEx, reinvestment, shareholder returns

### 🔮 Forecasting & Analyst Analytics
16. **Revenue Forecasting** - Estimate spreads, consensus uncertainty
17. **Analyst Sentiment** - Price target dynamics, recommendations (ENHANCED v1.15)

### 👥 Operations & Efficiency
18. **Employee Productivity** - Revenue per employee, hiring intensity
19. **Efficiency Ratios** - Asset turnover, inventory efficiency
20. **Balance Sheet Dynamics** - Asset/debt growth, working capital
21. **Employment Dynamics** - FTE growth, workforce volatility

### 📅 Temporal Analytics (NEW v1.15)
22. **Fiscal Calendar Features** - FY progress, quarter-end proximity, reporting lag
23. **Cash Flow Temporal** - FCF/CFO quarterly trends, investment intensity
24. **Temporal Patterns** - Earnings calendar, seasonality analysis

## v1.15 New Feature Generators
- `engineer_price_target_dynamics` - 35+ PT momentum/acceleration features
- `engineer_valuation_timeseries_features` - 22 valuation trend features
- `engineer_analyst_coverage_features` - 12 coverage trajectory features
- `engineer_revenue_forecast_features` - 15 estimate skew/alignment features
- `engineer_dividend_reliability_features` - 26 consistency/yield history features
- `engineer_employment_dynamics_features` - 10 workforce volatility features
- `engineer_employee_productivity_features` - 26 revenue/profit per employee features
- `engineer_balance_sheet_trends` - 13 working capital/asset stability features
- `engineer_margin_trends` - 6 margin consistency features
- `engineer_accounting_quality_features` - 18 exceptional items/pattern features
- `engineer_fiscal_calendar_features` - 9 fiscal timing features
- `engineer_eps_trajectory_features` - 14 EPS trend/CAGR features
- `engineer_cashflow_temporal_features` - 12 FCF/CFO temporal features

## Integrated Artifacts
- `outputs/eda/visualizations/` - Sector benchmarking, hypothesis tests
- `outputs/eda/earnings_analytics/` - Earnings surprises, market movers
- `outputs/eda/dividend_visualizations/` - Dividend yield by sector
- `outputs/eda/advanced_analytics/` - VaR, risk attribution
- `outputs/eda/price_target_dynamics/` - PT momentum analytics (NEW)
- `outputs/eda/eps_trajectory_analytics/` - EPS CAGR analysis (NEW)
- `outputs/eda/cashflow_temporal_analytics/` - FCF temporal trends (NEW)
- `outputs/eda/temporal_analytics/` - Fiscal calendar dashboards (NEW)


In [1]:
# ============================================================================
# Cell 1: Configuration & Setup
# ============================================================================
import json
import sys
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# SQLAlchemy check
try:
    from sqlalchemy import create_engine, text

    HAVE_SQLALCHEMY = True
except ImportError:
    HAVE_SQLALCHEMY = False

# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
CACHE_DIR = PROJECT_ROOT / '.cache'

# Create output directories
for subdir in ['eda/dashboards', 'eda/by_region', 'eda/by_sector',
               'eda/by_industry', 'eda/by_exchange']:
    (OUTPUT_DIR / subdir).mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from finance_ml.notebook_config import NotebookConfig
from finance_ml.core.schema import (
    PHASE93_FEATURE_CATEGORIES,
)

# Earnings Dashboard Widgets
from finance_ml.dashboards.widgets import (
    display_earnings_dashboard,
    create_earnings_metrics_chart,
    create_category_comparison_chart,
    create_earnings_surprise_dashboard,
    create_analyst_recommendation_heatmap,
    create_market_movers_dashboard,
    create_price_target_scorecard,
    create_earnings_calendar_analytics,
    analyze_earnings_quality,
    create_gaap_adjusted_comparison_chart,
    create_technical_valuation_dashboard,
    create_dividend_sustainability_scorecard,
    create_employee_productivity_scorecard,
    create_dividend_reliability_scorecard,
    create_leverage_liquidity_scorecard,
    create_analyst_consensus_scorecard,
    create_earnings_quality_scorecard,
    create_revenue_forecast_scorecard,
    create_category_correlation_network,
    generate_earnings_quality_alerts,
    EarningsAlertConfig,
    resolve_reference_date,
    add_formatted_date_columns,
    # NEW v1.15 dashboard functions
    create_price_target_dynamics_dashboard,
    create_eps_trajectory_dashboard,
    create_cashflow_temporal_dashboard,
    create_fiscal_calendar_dashboard,
)
from finance_ml.core.constants import (
    DATE_DISPLAY_FORMAT,
    PLOTLY_TEMPLATE,
    COLOR_PALETTE,
    CATEGORY_COLORS,
)

from finance_ml.features.advanced import (
    engineer_temporal_features,
    # NEW v1.15 feature generators
    engineer_price_target_dynamics,
    engineer_fiscal_calendar_features,
    engineer_dividend_timing_features,
    engineer_eps_trajectory_features,
    engineer_cashflow_temporal_features,
)
import finance_ml.ml_workflow.eda as eda

CFG = NotebookConfig(
    have_finance_prediction=True,
    have_database_connection=True,
    have_advanced_analytics=True,
    have_dim_reduction=True,
    debug_mode=False,
)

# Configuration Constants
from finance_ml.core.constants import (
    RANDOM_SEED, MODEL_VERSION,
)

np.random.seed(RANDOM_SEED)

# Visualization Configuration
TOP_N_SECTORS = 12
TOP_N_INDUSTRIES = 30
TOP_N_EXCHANGES = 25
TOP_N_FEATURES = 100

# Style Configuration (code_guidelines.md §17)
plt.style.use('dark_background')
sns.set_palette('husl')


def prepare_for_plotly(plotly_df: pd.DataFrame, plotly_columns: list[str] | None = None) -> pd.DataFrame:
    """Convert categorical columns to string for Plotly compatibility."""
    df_copy = plotly_df.copy()
    cols_to_convert = plotly_columns or df_copy.select_dtypes(include=['category']).columns.tolist()
    for col_name in cols_to_convert:
        if col_name in df_copy.columns:
            df_copy[col_name] = df_copy[col_name].astype(str)
    return df_copy


print('=' * 80)
print('STOCK ANALYTICS DASHBOARD - CONFIGURATION')
print('=' * 80)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')
print(f'Model Version: {MODEL_VERSION}')
print(f'Phase 9.3 Categories: {len(PHASE93_FEATURE_CATEGORIES)}')
print(f'Total Phase 9.3 Features: {sum(len(f) for f in PHASE93_FEATURE_CATEGORIES.values())}')
CFG.display_summary()

STOCK ANALYTICS DASHBOARD - CONFIGURATION
PROJECT_ROOT: C:\Users\markm\PycharmProjects\Finance_Analytics_Platform
Python: 3.14.2
Model Version: v9_11
Phase 9.3 Categories: 21
Total Phase 9.3 Features: 393
FEATURE FLAGS CONFIGURATION

Core Features:
  Financial Prediction:        ✓ Enabled
  Database Connection:         ✓ Enabled
  Advanced Analytics:          ✓ Enabled
  Dimensionality Reduction:    ✓ Enabled

Analysis Features:
  Sector Analysis:             ✓ Enabled
  Region Analysis:             ✓ Enabled

Output Features:
  Interactive Plots:           ✓ Enabled
  Excel Export:                ✓ Enabled
  Portfolio Optimization:      ✓ Enabled

Development:
  Debug Mode:                  ✗ Disabled


## Cell 2: ETL Pipeline & Data Loading


In [2]:
# ============================================================================
# Cell 2: ETL Pipeline - Extract, Transform, Load
# ============================================================================

from finance_ml.etl import (
    ETLConfig,
    DataExtractionConfig,
    SchemaValidationConfig,
    DtypeCastingConfig,
    SemanticClassificationConfig,
    ImputationConfig,
    CurrencyConversionConfig,
    SemanticTransformConfig,
    DataSanitizationConfig,
    ScalingConfig,
    FeatureEngineeringConfig,
    FeatureSelectionConfig,
    FinancialMetricsConfig,
    run_etl_pipeline,
    ETLPipeline,
    ETLMetrics
)
from finance_ml.ml_workflow.eda.phase93_categories import (
    categorize_dataframe_columns, get_phase93_coverage_stats, )

etl_config = ETLConfig(
    extraction=DataExtractionConfig(normalize_column_names=True),
    validation=SchemaValidationConfig(
        validate_schema=True, require_target_column=True,
        drop_rows_with_missing_critical_fields=True,
        validate_schema_alignment=True, schema_alignment_threshold=0.80,
    ),
    dtype_casting=DtypeCastingConfig(apply_dtype_casting=True, track_diagnostics=True),
    semantic_classification=SemanticClassificationConfig(enabled=True, preserve_price_columns=True),
    imputation=ImputationConfig(
        apply_imputation=False, strategy="6step", knn_neighbors=5,
        sector_column="sector", reference_price_column="last_price",
        impute_categorical_columns=True, impute_datetime_columns=True,
        # NEW: Business-rule imputation (v1.19)
        apply_dividend_zero_fill=True,
        apply_analyst_rating_zero_fill=True,
        apply_financial_statement_zero_fill=True,
    ),
    currency_conversion=CurrencyConversionConfig(
        enabled=False,
        target_currency='USD',
        suffix='_usd'
    ),
    semantic_transform=SemanticTransformConfig(
        apply_log_transforms=True, exclude_ratios_from_winsorization=True,
        exclude_percentages_from_winsorization=True, exclude_counts_from_scaling=True
    ),
    sanitization=DataSanitizationConfig(
        sanitize_data=False, apply_winsorization=False,
        # NEW: Business-rule sanitization (v1.19)
        apply_business_rule_zero_fills=True,
    ),
    scaling=ScalingConfig(enabled=False, exclude_price_columns=True),
    feature_engineering=FeatureEngineeringConfig(
        enabled=True, preset="comprehensive", engineer_earnings_analytics=True,
    ),
    feature_selection=FeatureSelectionConfig(enabled=False),
    financial_metrics=FinancialMetricsConfig(
        compute_valuation_metrics=True, compute_profitability_metrics=True,
        compute_growth_metrics=True, compute_leverage_metrics=True,
        compute_target_vs_price_metrics=True, compute_sector_specific_metrics=True, generate_quality_alerts=True,
        generate_metrics_dashboard=True
    ),
)

In [3]:
%%sql
SELECT * FROM public.equities all_stocks WHERE "Income Statement Report Date" > '01-01-2024'

,Ticker,ISIN,Name,Region,Country,Trading Country,Exchange,Unit,Sector,Industry,...,Merger & Restructuring Charges (FY),Merger & Restructuring Charges (5YAVGFQ),Description,Fiscal Month,Fiscal Quarter,Fiscal Year,Reporting Lag,Next Income Statement Report Date,Reporting Interval,Earnings Report (Frequency)
0,NVDA,US67066G1040,NVIDIA Corporation,United States and Canada,US,US,NasdaqGS,USD,Information Technology,Semiconductors and Semiconductor Equipment,...,0.0,-385.50,NVIDIA Corporation a computing infrastructure ...,9.0,3.0,2026.0,122,2026-07-26,1.0,Quarterly
1,GOOGL,US02079K3059,Alphabet Inc.,United States and Canada,US,US,NasdaqGS,USD,Communication Services,Interactive Media and Services,...,-1796.0,-1009.50,Alphabet Inc. offers various products and plat...,9.0,3.0,2025.0,127,2026-06-30,1.0,Quarterly
2,VSNT,US9252831030,Versant Media Group Inc.,United States and Canada,US,US,NasdaqGS,USD,Communication Services,Media,...,0.0,0.00,Versant Media Group Inc. focuses on operating ...,NaN,NaN,NaN,241,NaN,NaN,NaN
3,AAPL,US0378331005,Apple Inc.,United States and Canada,US,US,NasdaqGS,USD,Information Technology,Technology Hardware Storage and Peripherals,...,0.0,-5.00,Apple Inc. designs manufactures and markets sm...,12.0,4.0,2026.0,124,2026-09-27,1.0,Quarterly
4,MSFT,US5949181045,Microsoft Corporation,United States and Canada,US,US,NasdaqGS,USD,Information Technology,Software,...,0.0,-112.67,Microsoft Corporation develops and supports so...,3.0,1.0,2026.0,120,2025-12-30,0.0,Quarterly
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6695,CILEASING,NGCILEASING2,C & I Leasing Plc,Africa / Middle East,NG,NG,NGSE,NGN,Industrials,Trading Companies and Distributors,...,0.0,1.37,C & I Leasing Plc provides marine services in ...,9.0,3.0,2025.0,122,2026-06-30,1.0,Quarterly
6696,TCSA3,BRTCSAACNOR3,Tecnisa S.A.,Latin America and Caribbean,BR,BR,BOVESPA,BRL,Consumer Discretionary,Household Durables,...,0.0,0.00,Tecnisa S.A. develops and constructs residenti...,9.0,3.0,2025.0,171,2026-06-30,1.0,Quarterly
6697,SIAME,TN0006590012,Société Industrielle d'Appareillage et de Maté...,Africa / Middle East,TN,TN,BVMT,TND,Industrials,Electrical Equipment,...,0.0,0.00,Société Industrielle d'Appareillage et de Maté...,6.0,2.0,2025.0,312,2025-12-30,1.0,Semi-Annually
6698,BERGER,NGBERGER0000,Berger Paints Nigeria Plc,Africa / Middle East,NG,NG,NGSE,NGN,Materials,Chemicals,...,0.0,0.00,Berger Paints Nigeria Plc manufactures distrib...,9.0,3.0,2025.0,119,2026-06-30,1.0,Quarterly


In [4]:
# ============================================================================
# ETL PIPELINE EXECUTION
# ============================================================================

print('=' * 80)
print('ETL PIPELINE EXECUTION')
print('=' * 80)

# Use the DataFrame imported from the preceding SQL cell
# The SQL query: SELECT * FROM public.equities all_stocks WHERE "Income Statement Report Date" > '01-01-2024'
# already loaded data into `df`

from finance_ml.core.schema import normalize_column_name

# Normalize column names (SQL has mixed case, Python needs lowercase)
all_stocks.columns = [normalize_column_name(col) for col in all_stocks.columns]
print(f"✓ Normalized {len(all_stocks.columns)} column names from imported SQL data")

# Create ETL pipeline for transformation only (data already extracted via SQL cell)
pipeline = ETLPipeline(config=etl_config)

# Initialize metrics manually since we're using pre-loaded data
pipeline.metrics = ETLMetrics(source_type='all_stocks')
pipeline.metrics.rows_input = len(all_stocks)
pipeline.metrics.columns_input = len(all_stocks.columns)

# Transform and load the pre-imported DataFrame
df_transformed = pipeline.transform(all_stocks)
df_transformed = pipeline.load(df_transformed)

# Capture metrics
metrics = pipeline.metrics

print('\n' + metrics.summary())

# Resolve reference date
REFERENCE_DATE = resolve_reference_date(df_transformed, None)
print(f"\nReference date: {REFERENCE_DATE.strftime(DATE_DISPLAY_FORMAT)}")

# Get Phase 9.3 coverage stats
coverage_stats = get_phase93_coverage_stats(df_transformed)
total_phase93 = sum(coverage_stats.values())

print(f'\n✓ ETL Complete: {df_transformed.shape[0]:,} stocks × {df_transformed.shape[1]} features')
print(f'✓ Phase 9.3 Features: {total_phase93}/{sum(len(f) for f in PHASE93_FEATURE_CATEGORIES.values())}')

# Update the main DataFrame reference
df = df_transformed


ETL PIPELINE EXECUTION
✓ Normalized 416 column names from imported SQL data

ETL Pipeline Summary:\n  Source: all_stocks\n  Duration: 0.00s (extract: 0.00s, transform: 0.00s, load: 0.00s)\n  Data: 6700 → 0 rows, 416 → 0 columns\n  Dtype Casting: Applied (0 coercion warnings, 0 unknown columns) ✓\n  Financial Metrics: 22 added (valuation: 4, profitability: 5, growth: 3, leverage: 2)\n  Semantic Classification: ✓ (Price Columns: 52, Market Value: 260, Ratios: 72, Log-Transformed: 38)\n  Feature Engineering: comprehensive (406 features added)\n  Business Rules: ✓ (0 negative value violations sanitized, 172 log-transforms skipped)\n  Schema Validation: ✓ (alignment: 100.00%, unknown extra: 62, missing required: 0, dtype mismatches: 439, recognized: 767)\n  Quality: 0.957, Validation: 1.000

Reference date: 14 Jan 2026

✓ ETL Complete: 6,700 stocks × 1106 features
✓ Phase 9.3 Features: 315/393


In [5]:
df

,ticker,isin,name,region,country,trading_country,exchange,unit,sector,industry,...,piotroski_f_score,eps_improvement_count,eps_trajectory_score,eps_stability,share_dilution_rate,dilution_score,altman_z_score,beneish_m_score,composite_quality_score,momentum_score
0,NVDA,US67066G1040,NVIDIA Corporation,United States and Canada,US,US,NasdaqGS,USD,Information Technology,Semiconductors and Semiconductor Equipment,...,6,1,20.0,0.0,-0.016235,51.623482,5.027156,0.09953,100.000000,23.739392
1,GOOGL,US02079K3059,Alphabet Inc.,United States and Canada,US,US,NasdaqGS,USD,Communication Services,Interactive Media and Services,...,6,1,20.0,0.579553,-0.035874,53.587408,2.434902,-0.050652,92.500000,33.214201
2,VSNT,US9252831030,Versant Media Group Inc.,United States and Canada,US,US,NasdaqGS,USD,Communication Services,Media,...,5,0,0.0,<NA>,<NA>,<NA>,1.13809,-0.091158,85.000000,<NA>
3,AAPL,US0378331005,Apple Inc.,United States and Canada,US,US,NasdaqGS,USD,Information Technology,Technology Hardware Storage and Peripherals,...,4,1,20.0,0.763958,-0.033294,53.329429,2.266021,0.00147,100.000000,20.919743
4,MSFT,US5949181045,Microsoft Corporation,United States and Canada,US,US,NasdaqGS,USD,Information Technology,Software,...,4,0,0.0,0.718785,0.00001,49.999033,1.829353,-0.066201,100.000000,21.996767
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6695,CILEASING,NGCILEASING2,C & I Leasing Plc,Africa / Middle East,NG,NG,NGSE,NGN,Industrials,Trading Companies and Distributors,...,5,0,0.0,1.0,1.263358,0.0,0.638446,-0.272516,50.000000,24.067315
6696,TCSA3,BRTCSAACNOR3,Tecnisa S.A.,Latin America and Caribbean,BR,BR,BOVESPA,BRL,Consumer Discretionary,Household Durables,...,3,3,60.0,0.011681,0.0,50.0,-2.156763,-0.125866,50.000000,17.373884
6697,SIAME,TN0006590012,Société Industrielle d'Appareillage et de Maté...,Africa / Middle East,TN,TN,BVMT,TND,Industrials,Electrical Equipment,...,7,2,40.0,0.178416,0.0,50.0,1.907544,0.002465,87.222222,15.84523
6698,BERGER,NGBERGER0000,Berger Paints Nigeria Plc,Africa / Middle East,NG,NG,NGSE,NGN,Materials,Chemicals,...,8,0,0.0,1.0,0.0,50.0,3.45268,-0.001848,100.000000,28.743949


## Cell 3: Region & Exchange Distribution Analytics

Interactive dashboards analyzing stock distribution by Region, Sector, Industry, and Exchange.


In [6]:
# ============================================================================
# Cell 3: Region, Sector, Industry & Exchange Distribution
# ============================================================================

print('=' * 80)
print('📊 REGION, SECTOR, INDUSTRY & EXCHANGE ANALYTICS')
print('=' * 80)

dashboard_dir = OUTPUT_DIR / 'eda' / 'dashboards'

# ============================================================================
# 3.1 Region Distribution with Key Metrics
# ============================================================================
if 'region' in df.columns:
    print('\n📍 Region Distribution Analysis...')

    region_summary_stats = df.groupby('region').agg({
        'ticker': 'count',
        'market_cap': ['sum', 'mean', 'median'],
        'last_price': 'mean',

    }).round(2)
    region_summary_stats.columns = ['Count', 'Total_MCap', 'Mean_MCap', 'Median_MCap', 'Avg_Price']
    region_summary_stats = region_summary_stats.sort_values('Count', ascending=False)
    region_summary_stats['Pct'] = (region_summary_stats['Count'] / region_summary_stats['Count'].sum() * 100).round(1)

    # Create region sunburst with sector breakdown
    fig_region_sb = px.sunburst(
        prepare_for_plotly(df, ['region', 'sector']),
        path=['region', 'sector'], values='market_cap',
        title='<b>Market Cap Distribution by Region → Sector</b>',
        template=PLOTLY_TEMPLATE, color='region',
        color_discrete_sequence=px.colors.qualitative.Set2,
    )
    fig_region_sb.update_layout(height=700, font=dict(family='Segoe UI, Roboto, Arial'))
    fig_region_sb.write_html(dashboard_dir / 'region_sector_sunburst.html')
    fig_region_sb.show()

    print(f'  ✓ Saved: region_sector_sunburst.html')
    print(f'\n  Region Summary:')
    print(region_summary_stats.to_string())

# ============================================================================
# 3.2 Exchange Distribution Analysis (Enhanced)
# ============================================================================
exchange_col = None
for col in ['exchange', 'primary_exchange', 'stock_exchange']:
    if col in df.columns:
        exchange_col = col
        break

if exchange_col:
    print(f'\n🏛️ Exchange Distribution Analysis ({exchange_col})...')

    exchange_stats = df.groupby(exchange_col).agg({
        'ticker': 'count',
        'market_cap': ['sum', 'mean'],
        'p_e': 'mean',
        'last_price': 'mean',
    }).round(2)
    exchange_stats.columns = ['Count', 'Total_MCap', 'Mean_MCap', 'Avg_Price', 'Mean_P_E']
    exchange_stats = exchange_stats.sort_values('Total_MCap', ascending=False).head(TOP_N_EXCHANGES)
    exchange_stats['Pct'] = (exchange_stats['Count'] / df.shape[0] * 100).round(1)

    # Filter to top exchanges
    top_exchanges = exchange_stats.index.tolist()
    df_exchange = df[df[exchange_col].isin(top_exchanges)].copy()


    # =========================================================================
    # Build top 5 P/E names for each level of the treemap hierarchy
    # Path: region → sector → industry
    # =========================================================================

    def get_top5_pe_names(subset_df):
        """Get top 5 stock names by P/E from a subset DataFrame."""
        valid = subset_df.dropna(subset=['p_e', 'name'])
        valid = valid[valid['p_e'] > 0]  # Only positive P/E
        top5 = valid.nlargest(5, 'p_e')['name'].tolist()
        return ', '.join(top5) if top5 else 'N/A'


    # Level 1: By Region only
    region_top_names = df_exchange.groupby('region').apply(
        get_top5_pe_names, include_groups=False
    ).to_dict()

    # Level 2: By Region + Sector
    region_sector_top_names = df_exchange.groupby(['region', 'sector']).apply(
        get_top5_pe_names, include_groups=False
    ).to_dict()

    # Level 3: By Region + Sector + Industry (full path)
    region_sector_industry_top_names = df_exchange.groupby(
        ['region', 'sector', 'industry']
    ).apply(get_top5_pe_names, include_groups=False).to_dict()


    # Create a unified lookup function for the treemap path
    def get_top_names_for_path(row):
        """Return top names based on the most specific available path."""
        region = row.get('region')
        sector = row.get('sector')
        industry = row.get('industry')

        # Try most specific first (region + sector + industry)
        key_full = (region, sector, industry)
        if key_full in region_sector_industry_top_names:
            return region_sector_industry_top_names[key_full]

        # Fall back to region + sector
        key_rs = (region, sector)
        if key_rs in region_sector_top_names:
            return region_sector_top_names[key_rs]

        # Fall back to region only
        if region in region_top_names:
            return region_top_names[region]

        return 'N/A'


    # Add top names column (for industry-level hover)
    df_exchange['top_names_by_pe'] = df_exchange.apply(get_top_names_for_path, axis=1)

    # =========================================================================
    # Create treemap with customdata for all hierarchy levels
    # =========================================================================

    # Prepare data with all level lookups embedded
    treemap_df = prepare_for_plotly(df_exchange, [exchange_col, 'region', 'sector', 'industry'])

    # Exchange treemap: Region → Sector → Industry
    fig_exchange = px.treemap(
        treemap_df,
        path=['region', 'sector', 'industry'],
        values='p_e',
        color='p_e',
        color_continuous_scale='Blues',
        title=f'<b>P/E by Region → Sector → Industry</b><br><sup>Top {TOP_N_EXCHANGES} Exchanges | Hover for Top 5 Stocks by P/E</sup>',
        template=PLOTLY_TEMPLATE,
        hover_data={'top_names_by_pe': True},
    )

    # Custom hover template showing path-specific top names
    fig_exchange.update_traces(
        hovertemplate='<b>%{label}</b><br>P/E Sum: %{value:.2f}<br>Top 5 by P/E: %{customdata[0]}<extra></extra>'
    )
    fig_exchange.update_layout(height=800)
    fig_exchange.write_html(dashboard_dir / 'exchange_sector_treemap.html')
    fig_exchange.show()

    print(f'  ✓ Saved: exchange_sector_treemap.html')
    print(f'\n  Top Exchanges by Market Cap:')
    print(exchange_stats.head(10).to_string())

    # Debug: Show sample of computed top names per level
    print(f'\n  📊 Top 5 P/E Names by Hierarchy Level:')
    print(f'    Regions: {len(region_top_names)} groups')
    print(f'    Region+Sector: {len(region_sector_top_names)} groups')
    print(f'    Region+Sector+Industry: {len(region_sector_industry_top_names)} groups')

# ============================================================================
# 3.3 Industry Distribution Analysis
# ============================================================================
industry_col = None
for col in ['industry', 'industry_group', 'sub_industry']:
    if col in df.columns:
        industry_col = col
        break

if industry_col:
    print(f'\n🏭 Industry Distribution Analysis ({industry_col})...')

    industry_stats = df.groupby(industry_col).agg({
        'ticker': 'count',
        'market_cap': ['sum', 'mean'],
        'roe': 'mean' if 'roe' in df.columns else 'count',
        'p_e': 'mean',
    }).round(2)

    if 'roe' in df.columns:
        industry_stats.columns = ['Count', 'Total_MCap', 'Mean_MCap', 'Avg_ROE', 'Mean_P_E']
    else:
        industry_stats.columns = ['Count', 'Total_MCap', 'Mean_MCap', '_count', 'Mean_P_E']
        industry_stats = industry_stats.drop('_count', axis=1)

    industry_stats = industry_stats.sort_values('Count', ascending=False).head(TOP_N_INDUSTRIES)

    # Industry bar chart
    fig_industry = px.bar(
        industry_stats.reset_index().head(TOP_N_INDUSTRIES),
        x='Count', y=industry_col, orientation='h',
        title=f'<b>Stock Count by Industry</b><br><sup>Top {TOP_N_INDUSTRIES} Industries</sup>',
        template=PLOTLY_TEMPLATE, color='Total_MCap',
        color_continuous_scale='Viridis',
    )
    fig_industry.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
    fig_industry.write_html(dashboard_dir / 'industry_distribution.html')
    fig_industry.show()

    print(f'  ✓ Saved: industry_distribution.html')

# ============================================================================
# 3.4 Region × Exchange × Sector 3D Analysis
# ============================================================================
if 'region' in df.columns and exchange_col:
    print('\n🌐 Region × Exchange × Sector 3D Analysis...')

    # Create cross-tabulation
    cross_tab = pd.crosstab([df['region'], df['sector']], df[exchange_col])

    # Region-Exchange heatmap
    region_exchange = pd.crosstab(df['region'], df[exchange_col])

    fig_re_heatmap = px.imshow(
        region_exchange, title='<b>Stock Distribution: Region × Exchange</b>',
        template=PLOTLY_TEMPLATE, color_continuous_scale='RdYlGn',
        aspect='auto', text_auto=True,
    )
    fig_re_heatmap.update_layout(height=500)
    fig_re_heatmap.write_html(dashboard_dir / 'region_exchange_heatmap.html')
    fig_re_heatmap.show()

    print(f'  ✓ Saved: region_exchange_heatmap.html')

print('\n✓ Region, Sector, Industry & Exchange Analytics Complete')


📊 REGION, SECTOR, INDUSTRY & EXCHANGE ANALYTICS

📍 Region Distribution Analysis...


  ✓ Saved: region_sector_sunburst.html

  Region Summary:
                             Count   Total_MCap  Mean_MCap  Median_MCap  Avg_Price   Pct
region                                                                                  
Europe                        2068  17622052.28    8521.30       971.04     174.84  30.9
Asia / Pacific                2055  27978436.71   13614.81      5691.20   14868.27  30.7
United States and Canada      1790  62457439.47   34892.42      5064.36     110.29  26.7
Africa / Middle East           511   3529164.71    6906.39      1109.01     219.31   7.6
Latin America and Caribbean    276   1491605.28    5404.37      1483.37     804.82   4.1

🏛️ Exchange Distribution Analysis (exchange)...


  ✓ Saved: exchange_sector_treemap.html

  Top Exchanges by Market Cap:
          Count   Total_MCap  Mean_MCap  Avg_Price  Mean_P_E   Pct
exchange                                                          
NasdaqGS    653  38114666.97   58368.56      47.75    112.49   9.7
NYSE        934  24991662.24   26757.67      36.69    124.27  13.9
TSE         339   5354549.91   15795.13      25.61   5560.19   5.1
SEHK        221   4297068.19   19443.75      32.99     39.53   3.3
SHSE        420   4211231.37   10026.74      60.99     65.43   6.3
SZSE        373   3297823.12    8841.35      67.14     51.77   5.6
ENXTPA      229   2992308.90   13066.85      30.08     64.72   3.4
LSE         259   2779014.96   10729.79      28.19     10.85   3.9
TWSE         96   2455843.30   25581.70      42.95    580.64   1.4
XTRA        205   2291884.41   11179.92      35.34     62.73   3.1

  📊 Top 5 P/E Names by Hierarchy Level:
    Regions: 5 groups
    Region+Sector: 45 groups
    Region+Sector+Industry: 269 

  ✓ Saved: industry_distribution.html

🌐 Region × Exchange × Sector 3D Analysis...


  ✓ Saved: region_exchange_heatmap.html

✓ Region, Sector, Industry & Exchange Analytics Complete


## Cell 4: Phase 9.3 Feature Category Dashboards

Comprehensive dashboards for each of the 21 Phase 9.3 feature categories.


In [7]:
# ============================================================================
# Cell 4: Phase 9.3 Feature Category Dashboards
# ============================================================================

print('=' * 80)
print('📊 PHASE 9.3 FEATURE CATEGORY DASHBOARDS')
print('=' * 80)

category_viz_dir = OUTPUT_DIR / 'eda' / 'dashboards' / 'categories'
category_viz_dir.mkdir(parents=True, exist_ok=True)

# Get categorized features
categorized = categorize_dataframe_columns(df)

# ============================================================================
# 4.1 Category Coverage Overview
# ============================================================================
print('\n📋 Feature Category Coverage Analysis...')

coverage_data = []
for category, features in PHASE93_FEATURE_CATEGORIES.items():
    present = [f for f in features if f in df.columns]
    avg_completeness = 0
    if present:
        avg_completeness = df[present].notna().mean().mean() * 100

    coverage_data.append({
        'Category': category,
        'Expected': len(features),
        'Present': len(present),
        'Coverage_Pct': len(present) / len(features) * 100 if features else 0,
        'Completeness_Pct': avg_completeness,
        'Color': CATEGORY_COLORS.get(category, '#666666'),
    })

coverage_df = pd.DataFrame(coverage_data).sort_values('Expected', ascending=False)

# Coverage bar chart
fig_coverage = go.Figure()
fig_coverage.add_trace(go.Bar(
    y=coverage_df['Category'], x=coverage_df['Expected'],
    name='Expected', orientation='h', marker_color=COLOR_PALETTE['neutral'],
))
fig_coverage.add_trace(go.Bar(
    y=coverage_df['Category'], x=coverage_df['Present'],
    name='Present', orientation='h', marker_color=COLOR_PALETTE['success'],
))
fig_coverage.update_layout(
    title='<b>Phase 9.3 Feature Coverage by Category</b>',
    template=PLOTLY_TEMPLATE, height=700, barmode='overlay',
    xaxis_title='Feature Count', yaxis_title='Category',
)
fig_coverage.write_html(category_viz_dir / 'phase93_coverage_overview.html')
fig_coverage.show()
print(f'  ✓ Saved: phase93_coverage_overview.html')


# ============================================================================
# 4.2 Generate Dashboard for Each Major Category
# ============================================================================

def create_category_dashboard(dashboard_df, category_name, category_features, output_dir, group_cols=None):
    """Create comprehensive dashboard for a feature category."""
    # Filter features to only include numeric columns present in dashboard_df
    available_features = [f for f in category_features
                          if f in dashboard_df.columns and pd.api.types.is_numeric_dtype(dashboard_df[f])]

    if not available_features:
        print(f"  ⚠ No numeric features available for {category_name}")
        return None

    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            f'{category_name}: Distribution by Sector',
            f'{category_name}: Top Features Correlation',
            f'{category_name}: Regional Comparison',
            f'{category_name}: Key Metrics Box Plots',
        ],
        specs=[[{'type': 'bar'}, {'type': 'heatmap'}],
               [{'type': 'scatter'}, {'type': 'box'}]],
        vertical_spacing=0.12, horizontal_spacing=0.1,
    )

    # 1. Mean by Sector (top 10 sectors)
    if 'sector' in dashboard_df.columns and available_features:
        sector_means = dashboard_df.groupby('sector')[available_features[:5]].mean()
        top_sectors = dashboard_df['sector'].value_counts().head(10).index
        sector_means = sector_means.loc[sector_means.index.isin(top_sectors)]

        for i_feat, feat in enumerate(available_features[:3]):
            if feat in sector_means.columns:
                fig.add_trace(
                    go.Bar(x=sector_means.index, y=sector_means[feat], name=feat[:20]),
                    row=1, col=1
                )

    # 2. Correlation heatmap
    if len(available_features) >= 3:
        corr = dashboard_df[available_features[:10]].corr()
        fig.add_trace(
            go.Heatmap(z=corr.values, x=corr.columns, y=corr.index,
                       colorscale='RdBu_r', zmin=-1, zmax=1),
            row=1, col=2
        )

    # 3. Regional scatter (first 2 features)
    if 'region' in dashboard_df.columns and len(available_features) >= 2:
        for region_name in dashboard_df['region'].unique()[:5]:
            region_df_slice = dashboard_df[dashboard_df['region'] == region_name]
            fig.add_trace(
                go.Scatter(x=region_df_slice[available_features[0]], y=region_df_slice[available_features[1]],
                           mode='markers', name=str(region_name)[:15], opacity=0.6),
                row=2, col=1
            )

    # 4. Box plots
    if 'sector' in dashboard_df.columns and available_features:
        for feat in available_features[:2]:
            fig.add_trace(
                go.Box(y=dashboard_df[feat], x=dashboard_df['sector'], name=feat[:15]),
                row=2, col=2
            )

    fig.update_layout(
        title=f'<b>{category_name} Dashboard</b><br><sup>{len(available_features)} features available</sup>',
        template=PLOTLY_TEMPLATE, height=900, showlegend=True,
    )

    output_path_file = output_dir / f'{category_name.lower().replace(" ", "_").replace("&", "and")}_dashboard.html'
    fig.write_html(output_path_file)
    return output_path_file


# Generate dashboards for key categories
key_categories = [
    'Momentum & Technical', 'Valuation Ratios', 'Profitability',
    'Quality & Risk', 'Growth Metrics', 'Leverage & Liquidity',
    'Analyst Sentiment', 'Earnings Quality', 'Dividend Reliability',
    'Revenue Forecasting', 'Balance Sheet Dynamics', 'Technical Analysis',
    'Valuation Timeseries', 'Employment Dynamics',
]

for category in key_categories:
    if category in PHASE93_FEATURE_CATEGORIES:
        features = PHASE93_FEATURE_CATEGORIES[category]
        result = create_category_dashboard(df, category, features, category_viz_dir)
        if result:
            print(f'  ✓ Created: {result.name}')

# ============================================================================
# 4.3 Feature Category Correlation Network
# ============================================================================
print('\n🕸️ Creating Feature Category Correlation Network...')
fig_network = create_category_correlation_network(df, category_mapping=PHASE93_FEATURE_CATEGORIES,
                                                  output_dir=category_viz_dir)
if fig_network is not None:
    fig_network.show()
    print(f'  ✓ Saved: category_correlation_network.html')

print('\n✓ Phase 9.3 Category Dashboards Complete')


📊 PHASE 9.3 FEATURE CATEGORY DASHBOARDS

📋 Feature Category Coverage Analysis...


  ✓ Saved: phase93_coverage_overview.html
  ✓ Created: momentum_and_technical_dashboard.html
  ✓ Created: valuation_ratios_dashboard.html
  ✓ Created: profitability_dashboard.html
  ✓ Created: quality_and_risk_dashboard.html
  ✓ Created: growth_metrics_dashboard.html
  ✓ Created: leverage_and_liquidity_dashboard.html
  ✓ Created: analyst_sentiment_dashboard.html
  ✓ Created: earnings_quality_dashboard.html
  ✓ Created: dividend_reliability_dashboard.html
  ✓ Created: revenue_forecasting_dashboard.html
  ✓ Created: balance_sheet_dynamics_dashboard.html
  ✓ Created: technical_analysis_dashboard.html
  ✓ Created: valuation_timeseries_dashboard.html
  ✓ Created: employment_dynamics_dashboard.html

🕸️ Creating Feature Category Correlation Network...


  ✓ Saved: category_correlation_network.html

✓ Phase 9.3 Category Dashboards Complete


## Cell 5: Regional Benchmarking Dashboard

Comprehensive regional analysis with key financial metrics comparison.


In [8]:
# ============================================================================
# Cell 5: Regional Benchmarking Dashboard
# ============================================================================

print('=' * 80)
print('🌍 REGIONAL BENCHMARKING DASHBOARD')
print('=' * 80)

region_dir = OUTPUT_DIR / 'eda' / 'by_region'

if 'region' in df.columns:
    # Key metrics for regional comparison
    benchmark_metrics = [
        'roe', 'roa', 'p_e_ratio', 'debt_to_equity', 'price_momentum_1m',
        'earnings_quality_score', 'ema_trend_consistency', 'piotroski_f_score',
        'altman_z_score', 'dividend_yield', 'revenue_growth_yoy',
    ]
    available_metrics = [m for m in benchmark_metrics if m in df.columns]

    # ============================================================================
    # 5.1 Regional Statistics Table
    # ============================================================================
    print('\n📊 Computing Regional Statistics...')

    regional_metrics_stats = df.groupby('region')[available_metrics].agg(['mean', 'median', 'std', 'count'])
    regional_metrics_stats.columns = ['_'.join(col) for col in regional_metrics_stats.columns]

    # Save to JSON
    regional_stats_dict = regional_metrics_stats.to_dict()
    with open(region_dir / 'regional_statistics.json', 'w') as f:
        json.dump({k: {str(kk): vv for kk, vv in v.items()}
                   for k, v in regional_stats_dict.items()}, f, indent=2, default=str)
    print(f'  ✓ Saved: regional_statistics.json')

    # ============================================================================
    # 5.2 Regional Radar Charts
    # ============================================================================
    print('\n🎯 Creating Regional Radar Charts...')

    # Normalize metrics for radar chart
    radar_metrics = available_metrics[:8]  # Limit for readability
    regional_means = df.groupby('region')[radar_metrics].mean()

    # Z-score normalize for comparability
    regional_normalized = (regional_means - regional_means.mean()) / regional_means.std()
    regional_normalized = regional_normalized.fillna(0)

    fig_radar = go.Figure()
    for region in regional_normalized.index:
        fig_radar.add_trace(go.Scatterpolar(
            r=regional_normalized.loc[region].values.tolist() + [regional_normalized.loc[region].values[0]],
            theta=radar_metrics + [radar_metrics[0]],
            fill='toself', name=str(region), opacity=0.6,
        ))

    fig_radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[-2, 2])),
        title='<b>Regional Financial Profile Comparison</b><br><sup>Z-Score Normalized Metrics</sup>',
        template=PLOTLY_TEMPLATE, height=700,
    )
    fig_radar.write_html(region_dir / 'regional_radar_comparison.html')
    fig_radar.show()
    print(f'  ✓ Saved: regional_radar_comparison.html')

    # ============================================================================
    # 5.3 Regional Box Plots Grid
    # ============================================================================
    print('\n📦 Creating Regional Box Plot Grid...')

    n_metrics = min(6, len(available_metrics))
    rows = 2
    cols = 3

    fig_boxes = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=[m.replace('_', ' ').title() for m in available_metrics[:n_metrics]],
    )

    for idx, metric in enumerate(available_metrics[:n_metrics]):
        row = idx // cols + 1
        col = idx % cols + 1

        for i, region in enumerate(df['region'].unique()):
            region_data = df[df['region'] == region][metric].dropna()
            fig_boxes.add_trace(
                go.Box(y=region_data, name=str(region)[:10],
                       marker_color=px.colors.qualitative.Set2[i % 8],
                       showlegend=(idx == 0)),
                row=row, col=col
            )

    fig_boxes.update_layout(
        title='<b>Key Metrics Distribution by Region</b>',
        template=PLOTLY_TEMPLATE, height=700, showlegend=True,
    )
    fig_boxes.write_html(region_dir / 'regional_boxplots.html')
    fig_boxes.show()
    print(f'  ✓ Saved: regional_boxplots.html')

    # ============================================================================
    # 5.4 Regional Valuation Comparison (Statistical)
    # ============================================================================
    print('\n📊 Performing Statistical Regional Valuation Comparison...')
    valuation_metrics = ['p_e_ratio', 'p_b_ratio', 'ev_ebitda', 'dividend_yield']
    valuation_metrics = [m for m in valuation_metrics if m in df.columns]

    if valuation_metrics:
        regional_val_result = eda.compare_regional_valuations(
            df, metrics=valuation_metrics, region_column='region',
            include_tests=True, test_method='kruskal'
        )

        # The result is a dict with 'distributions' (DataFrame) and 'statistical_tests' (dict)
        distributions_df = regional_val_result['distributions']
        statistical_tests = regional_val_result['statistical_tests']

        # Create an interactive visualization from the distributions data
        fig_regional_val = px.bar(
            distributions_df,
            x='region',
            y='mean',
            color='metric',
            barmode='group',
            error_y='std',
            title='<b>Regional Valuation Comparison</b>',
            template=PLOTLY_TEMPLATE
        )
        fig_regional_val.show()
        fig_regional_val.write_html(region_dir / 'regional_valuation_comparison.html')
        print(f'  ✓ Saved: regional_valuation_comparison.html')

        # Also print statistical significance
        print("\n  📊 Statistical Test Results:")
        for metric, test_result in statistical_tests.items():
            sig = "✓ Significant" if test_result.get('significant') else "✗ Not significant"
            print(f"    {metric}: p-value={test_result.get('p_value', 0):.4f} ({sig})")

print('\n✓ Regional Benchmarking Complete')


🌍 REGIONAL BENCHMARKING DASHBOARD

📊 Computing Regional Statistics...
  ✓ Saved: regional_statistics.json

🎯 Creating Regional Radar Charts...


  ✓ Saved: regional_radar_comparison.html

📦 Creating Regional Box Plot Grid...


  ✓ Saved: regional_boxplots.html

📊 Performing Statistical Regional Valuation Comparison...


  ✓ Saved: regional_valuation_comparison.html

  📊 Statistical Test Results:
    p_e_ratio: p-value=0.0000 (✓ Significant)
    dividend_yield: p-value=0.0000 (✓ Significant)

✓ Regional Benchmarking Complete


## Cell 6: Sector Deep-Dive Analytics


In [9]:
# ============================================================================
# Cell 6: Sector Deep-Dive Analytics
# ============================================================================

print('=' * 80)
print('🏢 SECTOR DEEP-DIVE ANALYTICS')
print('=' * 80)

sector_dir = OUTPUT_DIR / 'eda' / 'by_sector'

if 'sector' in df.columns:
    # ============================================================================
    # 6.1 Sector Performance Heatmap
    # ============================================================================
    print('\n🔥 Creating Sector Performance Heatmap...')

    perf_metrics = [
        'roe', 'roa', 'price_momentum_1m', 'price_momentum_3m',
        'revenue_growth_yoy', 'earnings_quality_score', 'debt_to_equity',
    ]
    perf_metrics = [m for m in perf_metrics if m in df.columns]

    sector_perf = df.groupby('sector')[perf_metrics].mean()

    # Z-score normalize
    sector_perf_z = (sector_perf - sector_perf.mean()) / sector_perf.std()

    fig_heatmap = px.imshow(
        sector_perf_z.T, title='<b>Sector Performance Heatmap</b><br><sup>Z-Score Normalized</sup>',
        template=PLOTLY_TEMPLATE, color_continuous_scale='RdYlGn',
        aspect='auto', text_auto='.2f',
    )
    fig_heatmap.update_layout(height=500, xaxis_title='Sector', yaxis_title='Metric')
    fig_heatmap.write_html(sector_dir / 'sector_performance_heatmap.html')
    fig_heatmap.show()
    print(f'  ✓ Saved: sector_performance_heatmap.html')

    # ============================================================================
    # 6.2 Sector Feature Category Coverage
    # ============================================================================
    print('\n📋 Analyzing Feature Coverage by Sector...')

    sector_coverage = []
    for sector in df['sector'].unique():
        sector_df = df[df['sector'] == sector]
        for category in list(PHASE93_FEATURE_CATEGORIES.keys())[:10]:  # Top 10 categories
            features = PHASE93_FEATURE_CATEGORIES[category]
            available = [f for f in features if f in sector_df.columns]
            if available:
                completeness = sector_df[available].notna().mean().mean() * 100
                sector_coverage.append({
                    'Sector': sector,
                    'Category': category,
                    'Completeness': completeness,
                })

    coverage_matrix = pd.DataFrame(sector_coverage).pivot(
        index='Category', columns='Sector', values='Completeness'
    )

    fig_coverage = px.imshow(
        coverage_matrix,
        title='<b>Phase 9.3 Feature Completeness by Sector × Category</b>',
        template=PLOTLY_TEMPLATE, color_continuous_scale='RdYlGn',
        aspect='auto', text_auto='.0f',
    )
    fig_coverage.update_layout(height=600)
    fig_coverage.write_html(sector_dir / 'sector_category_coverage.html')
    fig_coverage.show()
    print(f'  ✓ Saved: sector_category_coverage.html')

    # ============================================================================
    # 6.3 Sector Valuation Scatter
    # ============================================================================
    print('\n💰 Creating Sector Valuation Scatter...')

    if 'p_e_ratio' in df.columns and 'roe' in df.columns:
        fig_scatter = px.scatter(
            df[df['p_e_ratio'].between(0, 100) & df['roe'].between(-50, 100)],
            x='roe', y='p_e_ratio', color='sector',
            size='market_cap', size_max=30,
            hover_data=['ticker', 'last_price', 'price_target'],
            title='<b>Sector Valuation: ROE vs P/E Ratio</b><br><sup>Size = Market Cap</sup>',
            template=PLOTLY_TEMPLATE,
        )
        fig_scatter.update_layout(height=700)
        fig_scatter.write_html(sector_dir / 'sector_valuation_scatter.html')
        fig_scatter.show()
        print(f'  ✓ Saved: sector_valuation_scatter.html')

    # ============================================================================
    # 6.4 Sector Distribution Comparison
    # ============================================================================
    print('\n📊 Comparing Sector Distributions...')
    dist_metrics = available_metrics.copy()
    dist_metrics = [m for m in dist_metrics if m in df.columns]

    if dist_metrics:
        # Get distribution statistics (returns DataFrame, not Figure)
        sector_dist_df = eda.compare_sector_distributions(df, metrics=dist_metrics, sector_column='sector')

        # Create a visualization from the distribution data
        import plotly.express as px

        fig_sector_dist = px.box(
            df.melt(id_vars=['sector'], value_vars=dist_metrics, var_name='metric', value_name='value'),
            x='sector',
            y='value',
            color='metric',
            facet_col='metric',
            facet_col_wrap=3,
            title='Sector Distribution Comparison'
        )
        fig_sector_dist.update_layout(showlegend=False)
        fig_sector_dist.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

        fig_sector_dist.show()
        fig_sector_dist.write_html(sector_dir / 'sector_distribution_comparison.html')
        print(f'  ✓ Saved: sector_distribution_comparison.html')

print('\n✓ Sector Deep-Dive Analytics Complete')


🏢 SECTOR DEEP-DIVE ANALYTICS

🔥 Creating Sector Performance Heatmap...


  ✓ Saved: sector_performance_heatmap.html

📋 Analyzing Feature Coverage by Sector...


  ✓ Saved: sector_category_coverage.html

💰 Creating Sector Valuation Scatter...


  ✓ Saved: sector_valuation_scatter.html

📊 Comparing Sector Distributions...


  ✓ Saved: sector_distribution_comparison.html

✓ Sector Deep-Dive Analytics Complete


## Cell 7: Earnings & Dividend Analytics (earnings_widgets.py Integration)

Comprehensive earnings and dividend dashboards using Phase 9.3 widgets.


In [10]:
# ============================================================================
# Cell 7: Earnings & Dividend Analytics
# ============================================================================

print('=' * 80)
print('📈 EARNINGS & DIVIDEND ANALYTICS')
print('=' * 80)

earnings_dir = OUTPUT_DIR / 'eda' / 'earnings_analytics'
dividend_dir = OUTPUT_DIR / 'eda' / 'dividend_visualizations'

# ============================================================================
# 7.1 Earnings Surprise Dashboard
# ============================================================================
print('\n📊 Creating Earnings Surprise Dashboard...')
fig_surprise = create_earnings_surprise_dashboard(df, reference_date=REFERENCE_DATE)
if fig_surprise is not None:
    fig_surprise.write_html(earnings_dir / 'earnings_surprise_analysis.html')
    fig_surprise.show()
    print(f'  ✓ Saved: earnings_surprise_analysis.html')

# ============================================================================
# 7.2 Analyst Recommendation Heatmap
# ============================================================================
print('\n🎯 Creating Analyst Recommendation Heatmap...')
fig_analyst = create_analyst_recommendation_heatmap(df)
if fig_analyst is not None:
    fig_analyst.write_html(earnings_dir / 'analyst_recommendations.html')
    fig_analyst.show()
    print(f'  ✓ Saved: analyst_recommendations.html')

# ============================================================================
# 7.2b Analyst Consensus Dashboard (New - Task 3)
# ============================================================================
print('\n🎯 Creating Analyst Consensus Scorecard...')
analyst_consensus_scorecard = create_analyst_consensus_scorecard(
    df,
    output_path=earnings_dir
)
if isinstance(analyst_consensus_scorecard, pd.DataFrame):
    display(analyst_consensus_scorecard.head())
    print(f'  ✓ Analyst Consensus Scorecard complete')

# ============================================================================
# 7.3 Market Movers Dashboard
# ============================================================================
print('\n📈 Creating Market Movers Dashboard...')
fig_movers = create_market_movers_dashboard(df, reference_date=REFERENCE_DATE)
if fig_movers is not None:
    fig_movers.write_html(earnings_dir / 'market_movers.html')
    fig_movers.show()
    print(f'  ✓ Saved: market_movers.html')

# ============================================================================
# 7.4 Price Target Analytics
# ============================================================================
print('\n🎯 Creating Price Target Scorecard...')
price_target_scorecard = create_price_target_scorecard(
    df,
    output_path=earnings_dir
)
if isinstance(price_target_scorecard, pd.DataFrame):
    display(price_target_scorecard.head())
    print(f'  ✓ Price Target Scorecard complete')

# ============================================================================
# 7.5 Technical & Valuation Dashboard
# ============================================================================
print('\n📊 Creating Technical Valuation Dashboard...')
fig_tech = create_technical_valuation_dashboard(df, output_dir=earnings_dir)
if fig_tech is not None:
    fig_tech.show()
    print(f'  ✓ Saved: technical_valuation_dashboard.html')

# ============================================================================
# 7.6 Dividend Sustainability Scorecard
# ============================================================================
print('\n💰 Creating Dividend Sustainability Scorecard...')
dividend_scorecard = create_dividend_sustainability_scorecard(
    df, 
    output_path=dividend_dir
)
if isinstance(dividend_scorecard, pd.DataFrame):
    display(dividend_scorecard.head())
    print(f"  ✓ Dividend Sustainability Scorecard complete")

# ============================================================================
# 7.6b Dividend Reliability Dashboard (New - Task 1)
# ============================================================================
print('\n💰 Creating Dividend Reliability Scorecard...')
dividend_reliability_scorecard = create_dividend_reliability_scorecard(
    df,
    output_path=dividend_dir
)
if isinstance(dividend_reliability_scorecard, pd.DataFrame):
    display(dividend_reliability_scorecard.head())
    print(f'  ✓ Dividend Reliability Scorecard complete')

# ============================================================================
# 7.7 Earnings Quality Alerts
# ============================================================================
print('\n⚠️ Generating Earnings Quality Alerts...')
alert_config = EarningsAlertConfig()
alerts_payload = generate_earnings_quality_alerts(df, config=alert_config, reference_date=REFERENCE_DATE)
with open(earnings_dir / 'earnings_alerts.json', 'w') as f:
    json.dump(alerts_payload, f, indent=2, default=str)
print(f"  ✓ Saved: earnings_alerts.json ({len(alerts_payload.get('alerts', []))} alerts)")

# ============================================================================
# 7.8 Earnings Calendar Dashboard (Styled Table)
# ============================================================================
print('\n📅 Creating Styled Earnings Calendar Dashboard...')
styler_dashboard = display_earnings_dashboard(df, mode="all", reference_date=REFERENCE_DATE, top_n=20)
if styler_dashboard is not None:
    # In Jupyter, displaying the styler shows the formatted table
    from IPython.display import display

    display(styler_dashboard)

# ============================================================================
# 7.9 Earnings Metrics Chart
# ============================================================================
print('\n📊 Creating Earnings Metrics Chart...')
fig_metrics = create_earnings_metrics_chart(df, metric_category="Profitability", reference_date=REFERENCE_DATE)
if fig_metrics is not None:
    fig_metrics.write_html(earnings_dir / 'earnings_metrics_profitability.html')
    fig_metrics.show()
    print(f'  ✓ Saved: earnings_metrics_profitability.html')

# ============================================================================
# 7.10 Comprehensive Earnings Calendar Analytics
# ============================================================================
print('\n📅 Generating Comprehensive Earnings Calendar Analytics...')
calendar_results = create_earnings_calendar_analytics(df, output_dir=earnings_dir, reference_date=REFERENCE_DATE)
if calendar_results is not None:
    if 'timeline_fig' in calendar_results:
        calendar_results['timeline_fig'].show()
    if 'heatmap_fig' in calendar_results:
        calendar_results['heatmap_fig'].show()
    print(f"  ✓ Generated earnings calendar analytics")

# ============================================================================
# 7.11 Earnings Quality Analytics
# ============================================================================
print('\n📊 Analyzing Earnings Quality...')
quality_metrics = analyze_earnings_quality(df)
if quality_metrics is not None and not quality_metrics.empty:
    print(f"  ✓ Quality analysis complete: {len(quality_metrics)} indicators")

# ============================================================================
# 7.12 GAAP vs Adjusted Comparison
# ============================================================================
print('\n📊 Creating GAAP vs Adjusted Comparison Chart...')
output_path_gaap = earnings_dir / 'gaap_vs_adjusted_comparison.html'
fig_gaap = create_gaap_adjusted_comparison_chart(df, output_path=output_path_gaap)
if fig_gaap is not None:
    fig_gaap.show()
    print(f'  ✓ Saved: gaap_vs_adjusted_comparison.html')

# ============================================================================
# 7.12.5 Surprise Momentum Analysis
# ============================================================================
if 'surprise_momentum_score' in df.columns:
    print('\n📊 Analyzing Surprise Momentum...')
    fig_mom = px.histogram(
        df,
        x='surprise_momentum_score',
        color='sector',
        title='<b>Distribution of Earnings Surprise Momentum</b>',
        marginal='box',
        template=PLOTLY_TEMPLATE
    )
    fig_mom.write_html(earnings_dir / 'surprise_momentum_distribution.html')
    fig_mom.show()
    print(f'  ✓ Saved: surprise_momentum_distribution.html')

# ============================================================================
# 7.13 Category Comparison Chart
# ============================================================================
print('\n📊 Creating Category Comparison Chart...')
fig_cat_comp = create_category_comparison_chart(df, reference_date=REFERENCE_DATE)
if fig_cat_comp is not None:
    fig_cat_comp.show()
    fig_cat_comp.write_html(earnings_dir / 'category_comparison_chart.html')
    print(f'  ✓ Saved: category_comparison_chart.html')

# ============================================================================
# 7.14 Earnings Quality Dashboard (New - Task 5)
# ============================================================================
print('\n📊 Creating Earnings Quality Scorecard...')
earnings_quality_scorecard = create_earnings_quality_scorecard(
    df,
    output_path=earnings_dir
)
if isinstance(earnings_quality_scorecard, pd.DataFrame):
    display(earnings_quality_scorecard.head())
    print(f'  ✓ Earnings Quality Scorecard complete')

# ============================================================================
# 7.15 Leverage & Liquidity Heatmap (New - Task 2)
# ============================================================================
print('\n📊 Creating Leverage & Liquidity Scorecard...')
leverage_liquidity_scorecard = create_leverage_liquidity_scorecard(
    df,
    output_path=earnings_dir
)
if isinstance(leverage_liquidity_scorecard, pd.DataFrame):
    display(leverage_liquidity_scorecard.head())
    print(f'  ✓ Leverage & Liquidity Scorecard complete')

# ============================================================================
# 7.16 Revenue Forecast Momentum Chart (New - Task 6)
# ============================================================================
print('\n📊 Creating Revenue Forecast Scorecard...')
revenue_forecast_scorecard = create_revenue_forecast_scorecard(
    df,
    output_path=earnings_dir
)
if isinstance(revenue_forecast_scorecard, pd.DataFrame):
    display(revenue_forecast_scorecard.head())
    print(f'  ✓ Revenue Forecast Scorecard complete')

print('\n✓ Earnings & Dividend Analytics Complete')

📈 EARNINGS & DIVIDEND ANALYTICS

📊 Creating Earnings Surprise Dashboard...


  ✓ Saved: earnings_surprise_analysis.html

🎯 Creating Analyst Recommendation Heatmap...


  ✓ Saved: analyst_recommendations.html

🎯 Creating Analyst Consensus Scorecard...


,ticker,sector,region,rating_score,upside_score,consensus_strength_score,conviction_score,pt_momentum_score,analyst_consensus_score,consensus_grade
0,NVDA,Information Technology,United States and Canada,50.0,50.0,15.2,92.063492,50.018714,49.4,D
1,GOOGL,Communication Services,United States and Canada,50.0,50.0,27.352941,88.059701,50.082171,51.2,C
2,VSNT,Communication Services,United States and Canada,50.0,50.0,55.294118,0.0,50.0,43.6,D
3,AAPL,Information Technology,United States and Canada,50.0,50.0,55.0,51.020408,50.01093,51.2,C
4,MSFT,Information Technology,United States and Canada,50.0,50.0,60.917722,96.491228,49.98654,59.2,C


  ✓ Analyst Consensus Scorecard complete

📈 Creating Market Movers Dashboard...


...

  ✓ Earnings Quality Scorecard complete

📊 Creating Leverage & Liquidity Scorecard...


,ticker,sector,region,current_ratio_score,quick_ratio_score,cash_ratio_score,debt_equity_score,debt_assets_score,interest_coverage_score,leverage_liquidity_score,leverage_grade
0,NVDA,Information Technology,United States and Canada,50.0,50.0,50.0,96.966001,91.605543,0.0,57.7,C
1,GOOGL,Communication Services,United States and Canada,50.0,50.0,50.0,96.192059,89.702341,0.0,57.2,C
2,VSNT,Communication Services,United States and Canada,50.0,50.0,50.0,100.0,100.0,50.0,70.0,B
3,AAPL,Information Technology,United States and Canada,50.0,50.0,50.0,49.196425,60.897768,50.0,52.0,C
4,MSFT,Information Technology,United States and Canada,50.0,50.0,50.0,88.948595,76.354441,0.0,53.1,C


  ✓ Leverage & Liquidity Scorecard complete

📊 Creating Revenue Forecast Scorecard...


,ticker,sector,region,revenue_growth_score,est_revision_1m_score,est_revision_3m_score,revenue_beat_score,revenue_momentum_score,momentum_grade
0,NVDA,Information Technology,United States and Canada,100.0,50.0100,50.136667,0,57.5,C
1,GOOGL,Communication Services,United States and Canada,100.0,50.0065,50.209667,0,57.5,C
2,VSNT,Communication Services,United States and Canada,0.0,50.0000,50.000000,100,42.5,D
3,AAPL,Information Technology,United States and Canada,100.0,50.0005,50.101000,0,57.5,C
4,MSFT,Information Technology,United States and Canada,100.0,50.0075,50.155000,0,57.5,C


  ✓ Revenue Forecast Scorecard complete

✓ Earnings & Dividend Analytics Complete


## Cell 7.17: Price Target Dynamics Dashboard (NEW v1.15)

Analyze price target temporal dynamics including momentum, acceleration,
and consensus convergence patterns from the 15 new Analyst Sentiment features.


In [11]:
# ============================================================================
# Cell 7.17: Price Target Dynamics Dashboard (Phase 9.3 v1.15)
# ============================================================================

print('=' * 80)
print('📊 PRICE TARGET DYNAMICS ANALYTICS (v1.15)')
print('=' * 80)

pt_dynamics_dir = OUTPUT_DIR / 'eda' / 'price_target_dynamics'
pt_dynamics_dir.mkdir(parents=True, exist_ok=True)

# Engineer price target dynamics features if not present
pt_features = [
    'pt_momentum_1w', 'pt_momentum_1m', 'pt_momentum_3m', 'pt_momentum_6m',
    'pt_acceleration_short', 'pt_acceleration_long', 'pt_consensus_convergence',
    'analyst_coverage_change_1m', 'pt_vs_price_momentum'
]
missing_pt = [f for f in pt_features if f not in df.columns]

if missing_pt:
    print(f'\n🔧 Engineering {len(missing_pt)} price target dynamics features...')
    df = engineer_price_target_dynamics(df)

# Generate integrated dashboard
print('\n📊 Generating Price Target Dynamics Dashboard...')
fig_pt_dynamics = create_price_target_dynamics_dashboard(
    df, 
    output_path=pt_dynamics_dir / 'price_target_dynamics_dashboard.html'
)
if fig_pt_dynamics is not None:
    fig_pt_dynamics.show()
    print(f'  ✓ Saved: price_target_dynamics_dashboard.html')

# PT Momentum Heatmap by Sector
available_momentum = [f for f in ['pt_momentum_1w', 'pt_momentum_1m', 'pt_momentum_3m', 
                                   'pt_momentum_6m', 'pt_momentum_1y'] if f in df.columns]
if available_momentum and 'sector' in df.columns:
    print('\n📈 Generating PT Momentum Sector Heatmap...')
    
    top_sectors = df['sector'].value_counts().head(TOP_N_SECTORS).index.tolist()
    momentum_data = []
    
    for sector in top_sectors:
        sector_df = df[df['sector'] == sector]
        row = {'Sector': str(sector)[:25]}
        for feat in available_momentum:
            median_val = sector_df[feat].median()
            row[feat.replace('pt_momentum_', '').upper()] = float(median_val) if pd.notna(median_val) else 0
        momentum_data.append(row)
    
    momentum_df = pd.DataFrame(momentum_data).set_index('Sector')
    
    fig_pt_heatmap = px.imshow(
        momentum_df,
        title='<b>Price Target Momentum by Sector</b><br><sup>Median % Change Across Time Horizons</sup>',
        template=PLOTLY_TEMPLATE,
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        aspect='auto',
        text_auto='.1f',
    )
    fig_pt_heatmap.update_layout(height=600)
    fig_pt_heatmap.write_html(pt_dynamics_dir / 'pt_momentum_sector_heatmap.html')
    fig_pt_heatmap.show()
    print('  ✓ Saved: pt_momentum_sector_heatmap.html')

print('\n✓ Price Target Dynamics Analytics Complete')


📊 PRICE TARGET DYNAMICS ANALYTICS (v1.15)

📊 Generating Price Target Dynamics Dashboard...


  ✓ Saved: price_target_dynamics_dashboard.html

📈 Generating PT Momentum Sector Heatmap...


  ✓ Saved: pt_momentum_sector_heatmap.html

✓ Price Target Dynamics Analytics Complete


## Cell 7.18: EPS Trajectory Dashboard (NEW v1.15)

Deep dive into EPS trajectory patterns including quarterly trends,
growth acceleration, and multi-year CAGR analysis from 10 new features.


In [12]:
# ============================================================================
# Cell 7.18: EPS Trajectory Dashboard (Phase 9.3 v1.15)
# ============================================================================

print('=' * 80)
print('📊 EPS TRAJECTORY & MULTI-PERIOD ANALYSIS (v1.15)')
print('=' * 80)

eps_traj_dir = OUTPUT_DIR / 'eda' / 'eps_trajectory_analytics'
eps_traj_dir.mkdir(parents=True, exist_ok=True)

# Engineer EPS trajectory features if not present
eps_features = [
    'eps_quarterly_trend', 'eps_quarterly_volatility', 'eps_yoy_quarterly_growth',
    'eps_qoq_growth', 'eps_positive_streak', 'eps_cagr_5y', 'eps_cagr_3y',
    'eps_annual_trend', 'eps_vs_5y_avg', 'eps_growth_acceleration'
]
missing_eps = [f for f in eps_features if f not in df.columns]

if missing_eps:
    print(f'\n🔧 Engineering {len(missing_eps)} EPS trajectory features...')
    df = engineer_eps_trajectory_features(df)

# Generate integrated dashboard
print('\n📊 Generating EPS Trajectory Dashboard...')
fig_eps_traj = create_eps_trajectory_dashboard(
    df, 
    output_path=eps_traj_dir / 'eps_trajectory_dashboard.html'
)
if fig_eps_traj is not None:
    fig_eps_traj.show()
    print(f'  ✓ Saved: eps_trajectory_dashboard.html')

# EPS CAGR Comparison (3Y vs 5Y)
if 'eps_cagr_3y' in df.columns and 'eps_cagr_5y' in df.columns:
    print('\n📈 Generating EPS CAGR Comparison...')
    
    cagr_df = df[['ticker', 'sector', 'eps_cagr_3y', 'eps_cagr_5y']].dropna()
    cagr_df['cagr_3y_pct'] = cagr_df['eps_cagr_3y'] * 100
    cagr_df['cagr_5y_pct'] = cagr_df['eps_cagr_5y'] * 100
    
    fig_cagr = px.scatter(
        cagr_df.head(2000),
        x='cagr_5y_pct', y='cagr_3y_pct', color='sector',
        hover_data=['ticker'],
        title='<b>EPS CAGR: 3-Year vs 5-Year Comparison</b>',
        template=PLOTLY_TEMPLATE,
        opacity=0.7,
    )
    fig_cagr.add_shape(type='line', x0=-50, y0=-50, x1=100, y1=100,
                       line=dict(dash='dash', color='white', width=1))
    fig_cagr.update_layout(height=600, xaxis_title='5Y CAGR %', yaxis_title='3Y CAGR %')
    fig_cagr.write_html(eps_traj_dir / 'eps_cagr_comparison.html')
    fig_cagr.show()
    print('  ✓ Saved: eps_cagr_comparison.html')

# EPS Positive Streak Distribution
if 'eps_positive_streak' in df.columns:
    print('\n📊 Analyzing EPS Positive Streak...')
    
    fig_streak = px.histogram(
        df, x='eps_positive_streak', color='sector',
        title='<b>EPS Positive Streak Distribution</b><br><sup>Consecutive Quarters of Positive EPS</sup>',
        template=PLOTLY_TEMPLATE,
    )
    fig_streak.update_layout(height=500)
    fig_streak.write_html(eps_traj_dir / 'eps_positive_streak.html')
    fig_streak.show()
    print('  ✓ Saved: eps_positive_streak.html')

print('\n✓ EPS Trajectory Analytics Complete')


📊 EPS TRAJECTORY & MULTI-PERIOD ANALYSIS (v1.15)

📊 Generating EPS Trajectory Dashboard...


  ✓ Saved: eps_trajectory_dashboard.html

📈 Generating EPS CAGR Comparison...


  ✓ Saved: eps_cagr_comparison.html

📊 Analyzing EPS Positive Streak...


  ✓ Saved: eps_positive_streak.html

✓ EPS Trajectory Analytics Complete


## Cell 7.19: Cash Flow Temporal Patterns (NEW v1.15)

Analyze FCF/CFO temporal patterns including quarterly trends,
investment intensity, and acquisition activity from 12 new features.


In [13]:
# ============================================================================
# Cell 7.19: Cash Flow Temporal Dashboard (Phase 9.3 v1.15)
# ============================================================================

print('=' * 80)
print('📊 CASH FLOW TEMPORAL PATTERNS (v1.15)')
print('=' * 80)

cf_temporal_dir = OUTPUT_DIR / 'eda' / 'cashflow_temporal_analytics'
cf_temporal_dir.mkdir(parents=True, exist_ok=True)

# Engineer cash flow temporal features if not present
cf_features = [
    'fcf_quarterly_trend', 'fcf_quarterly_volatility', 'fcf_positive_ratio',
    'cfo_quarterly_trend', 'cfo_yoy_quarterly', 'investment_intensity_trend',
    'cfo_5y_trend', 'cfo_5y_stability', 'cfo_margin_current', 'cfo_margin_trend'
]
missing_cf = [f for f in cf_features if f not in df.columns]

if missing_cf:
    print(f'\n🔧 Engineering {len(missing_cf)} cash flow temporal features...')
    df = engineer_cashflow_temporal_features(df)

# Generate integrated dashboard
print('\n📊 Generating Cash Flow Temporal Dashboard...')
fig_cf_temporal = create_cashflow_temporal_dashboard(
    df, 
    output_path=cf_temporal_dir / 'cashflow_temporal_dashboard.html'
)
if fig_cf_temporal is not None:
    fig_cf_temporal.show()
    print(f'  ✓ Saved: cashflow_temporal_dashboard.html')

# FCF Health Matrix
if 'fcf_quarterly_trend' in df.columns and 'fcf_positive_ratio' in df.columns:
    print('\n📈 Generating FCF Health Matrix...')
    
    fcf_df = df[['ticker', 'sector', 'fcf_quarterly_trend', 'fcf_positive_ratio']].dropna()
    
    fig_fcf = px.scatter(
        fcf_df.head(2000),
        x='fcf_positive_ratio', y='fcf_quarterly_trend',
        color='sector', hover_data=['ticker'],
        title='<b>FCF Health: Trend vs Consistency</b><br><sup>X: Positive FCF Ratio, Y: Quarterly Trend</sup>',
        template=PLOTLY_TEMPLATE,
        opacity=0.7,
    )
    fig_fcf.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.5)
    fig_fcf.add_vline(x=0.5, line_dash='dash', line_color='white', opacity=0.5)
    fig_fcf.update_layout(height=600)
    fig_fcf.write_html(cf_temporal_dir / 'fcf_health_matrix.html')
    fig_fcf.show()
    print('  ✓ Saved: fcf_health_matrix.html')

print('\n✓ Cash Flow Temporal Analytics Complete')


📊 CASH FLOW TEMPORAL PATTERNS (v1.15)

📊 Generating Cash Flow Temporal Dashboard...


  ✓ Saved: cashflow_temporal_dashboard.html

📈 Generating FCF Health Matrix...


  ✓ Saved: fcf_health_matrix.html

✓ Cash Flow Temporal Analytics Complete


## Cell 7.20: Fiscal Calendar & Dividend Timing (NEW v1.15)

Analyze fiscal calendar patterns and dividend timing cycles from
17 new Temporal and Dividend Reliability features.


In [14]:
# ============================================================================
# Cell 7.20: Fiscal Calendar & Dividend Timing (Phase 9.3 v1.15)
# ============================================================================

print('=' * 80)
print('📅 FISCAL CALENDAR & DIVIDEND TIMING (v1.15)')
print('=' * 80)

temporal_dir = OUTPUT_DIR / 'eda' / 'temporal_analytics'
temporal_dir.mkdir(parents=True, exist_ok=True)

# Engineer fiscal calendar features if not present
fiscal_features = [
    'fiscal_year_progress', 'days_to_quarter_end', 'fiscal_half',
    'reporting_lag_zscore', 'late_reporter_flag', 'earnings_imminent'
]
missing_fiscal = [f for f in fiscal_features if f not in df.columns]

if missing_fiscal:
    print(f'\n🔧 Engineering {len(missing_fiscal)} fiscal calendar features...')
    df = engineer_fiscal_calendar_features(df, reference_date=REFERENCE_DATE)

# Engineer dividend timing features
div_timing_features = [
    'days_to_dividend_ex_date', 'approaching_ex_date', 'recently_ex_dividend',
    'dividend_cycle_position', 'dividend_announcement_recency'
]
missing_div_timing = [f for f in div_timing_features if f not in df.columns]

if missing_div_timing:
    print(f'\n🔧 Engineering {len(missing_div_timing)} dividend timing features...')
    df = engineer_dividend_timing_features(df, reference_date=REFERENCE_DATE)

# Generate integrated dashboard
print('\n📊 Generating Fiscal Calendar Dashboard...')
fig_fiscal = create_fiscal_calendar_dashboard(
    df, 
    output_path=temporal_dir / 'fiscal_calendar_dashboard.html'
)
if fig_fiscal is not None:
    fig_fiscal.show()
    print(f'  ✓ Saved: fiscal_calendar_dashboard.html')

# Fiscal Year Progress by Sector
if 'fiscal_year_progress' in df.columns and 'sector' in df.columns:
    print('\n📈 Analyzing Fiscal Year Progress by Sector...')
    
    fig_fy = px.box(
        df, x='sector', y='fiscal_year_progress',
        title='<b>Fiscal Year Progress by Sector</b><br><sup>0 = FY Start, 1 = FY End</sup>',
        template=PLOTLY_TEMPLATE,
        color='sector',
    )
    fig_fy.update_layout(height=500, showlegend=False, xaxis_tickangle=45)
    fig_fy.write_html(temporal_dir / 'fiscal_year_progress_by_sector.html')
    fig_fy.show()
    print('  ✓ Saved: fiscal_year_progress_by_sector.html')

# Earnings Window Summary
if 'earnings_imminent' in df.columns or 'pre_earnings_window' in df.columns:
    print('\n⏰ Analyzing Earnings Windows...')
    
    window_stats = {}
    if 'earnings_imminent' in df.columns:
        window_stats['Imminent (≤14d)'] = int(df['earnings_imminent'].sum())
    if 'pre_earnings_window' in df.columns:
        window_stats['Pre-Earnings (≤30d)'] = int(df['pre_earnings_window'].sum())
    
    window_df = pd.DataFrame([{'Window': k, 'Count': v} for k, v in window_stats.items()])
    
    fig_window = px.bar(
        window_df, x='Window', y='Count', color='Count',
        color_continuous_scale='Reds', text='Count',
        title='<b>Stocks in Earnings Windows</b>',
        template=PLOTLY_TEMPLATE,
    )
    fig_window.update_traces(textposition='outside')
    fig_window.update_layout(height=400)
    fig_window.write_html(temporal_dir / 'earnings_window_summary.html')
    fig_window.show()
    print('  ✓ Saved: earnings_window_summary.html')

# Dividend Cycle Analysis
if 'dividend_cycle_position' in df.columns:
    print('\n💰 Analyzing Dividend Cycle Positions...')
    
    fig_div_cycle = px.histogram(
        df[df['dividend_cycle_position'].notna()],
        x='dividend_cycle_position',
        title='<b>Dividend Cycle Position Distribution</b><br><sup>0 = Just Paid, 1 = Next Payment Due</sup>',
        template=PLOTLY_TEMPLATE,
        nbins=20,
    )
    fig_div_cycle.update_layout(height=400)
    fig_div_cycle.write_html(temporal_dir / 'dividend_cycle_distribution.html')
    fig_div_cycle.show()
    print('  ✓ Saved: dividend_cycle_distribution.html')

print('\n✓ Fiscal Calendar & Dividend Timing Analytics Complete')


📅 FISCAL CALENDAR & DIVIDEND TIMING (v1.15)

📊 Generating Fiscal Calendar Dashboard...


  ✓ Saved: fiscal_calendar_dashboard.html

📈 Analyzing Fiscal Year Progress by Sector...


  ✓ Saved: fiscal_year_progress_by_sector.html

⏰ Analyzing Earnings Windows...


  ✓ Saved: earnings_window_summary.html

💰 Analyzing Dividend Cycle Positions...


  ✓ Saved: dividend_cycle_distribution.html

✓ Fiscal Calendar & Dividend Timing Analytics Complete


## Cell 8: Employee Productivity & Efficiency Analytics


In [15]:
# ============================================================================
# Cell 8: Employee Productivity & Efficiency Analytics
# ============================================================================

print('=' * 80)
print('👥 EMPLOYEE PRODUCTIVITY & EFFICIENCY ANALYTICS')
print('=' * 80)

employment_dir = OUTPUT_DIR / 'eda' / 'employment_analytics'

# ============================================================================
# 8.1 Employee Productivity Dashboard
# ============================================================================
print('\n📊 Creating Employee Productivity Scorecard...')
employee_scorecard = create_employee_productivity_scorecard(
    df, 
    output_path=employment_dir
)
if isinstance(employee_scorecard, pd.DataFrame):
    display(employee_scorecard.head())
    print(f'  ✓ Employee Productivity Scorecard complete')

# ============================================================================
# 8.2 Efficiency Metrics by Sector
# ============================================================================
efficiency_metrics = [
    'asset_turnover', 'inventory_turnover', 'receivables_turnover',
    'revenue_per_employee', 'profit_per_employee',
]
efficiency_available = [m for m in efficiency_metrics if m in df.columns]

if efficiency_available and 'sector' in df.columns:
    print('\n⚙️ Analyzing Efficiency Metrics by Sector...')

    sector_efficiency = df.groupby('sector')[efficiency_available].mean()

    fig_eff = px.bar(
        sector_efficiency.reset_index().melt(id_vars='sector', var_name='Metric', value_name='Value'),
        x='sector', y='Value', color='Metric', barmode='group',
        title='<b>Efficiency Metrics by Sector</b>',
        template=PLOTLY_TEMPLATE,
    )
    fig_eff.update_layout(height=500, xaxis_tickangle=45)
    fig_eff.write_html(employment_dir / 'efficiency_by_sector.html')
    fig_eff.show()
    print(f'  ✓ Saved: efficiency_by_sector.html')

print('\n✓ Employee Productivity & Efficiency Analytics Complete')


👥 EMPLOYEE PRODUCTIVITY & EFFICIENCY ANALYTICS

📊 Creating Employee Productivity Scorecard...


,ticker,sector,region,revenue_per_emp_score,profit_per_emp_score,ebitda_per_emp_score,emp_growth_score,hiring_intensity_score,employee_productivity_score,productivity_grade
0,NVDA,Information Technology,United States and Canada,50.0,99.358209,99.179104,50.0,0.195244,67.2,B
1,GOOGL,Communication Services,United States and Canada,50.0,98.164179,97.268657,50.0,0.006771,66.5,B
2,VSNT,Communication Services,United States and Canada,50.0,21.186567,15.440299,50.0,50.0,35.9,D
3,AAPL,Information Technology,United States and Canada,50.0,98.477612,97.492537,50.0,0.0,66.6,B
4,MSFT,Information Technology,United States and Canada,50.0,97.865672,97.059701,50.0,0.0,66.4,B


  ✓ Employee Productivity Scorecard complete

⚙️ Analyzing Efficiency Metrics by Sector...


  ✓ Saved: efficiency_by_sector.html

✓ Employee Productivity & Efficiency Analytics Complete


## Cell 9: Hypothesis Testing & Statistical Benchmarking


In [16]:
# ============================================================================
# Cell 9: Hypothesis Testing & Statistical Benchmarking
# ============================================================================

print('=' * 80)
print('📊 HYPOTHESIS TESTING & STATISTICAL BENCHMARKING')
print('=' * 80)

import scipy.stats as scipy_stats

stats_dir = OUTPUT_DIR / 'eda' / 'advanced_analytics'

# ============================================================================
# 9.1 ANOVA Tests by Sector
# ============================================================================
test_metrics = ['roe', 'price_momentum_1m', 'debt_to_equity', 'earnings_quality_score', 'surprise_momentum_score']
test_metrics = [m for m in test_metrics if m in df.columns]

if test_metrics and 'sector' in df.columns:
    print('\n📈 Running ANOVA Tests by Sector...')

    anova_results = []
    for metric in test_metrics:
        groups = [group[metric].dropna().values for name, group in df.groupby('sector')]
        groups = [g for g in groups if len(g) >= 5]  # Min samples

        if len(groups) >= 2:
            f_stat, p_value = scipy_stats.f_oneway(*groups)
            anova_results.append({
                'Metric': metric,
                'F_Statistic': round(f_stat, 4),
                'P_Value': round(p_value, 6),
                'Significant': p_value < 0.05,
            })

    anova_df = pd.DataFrame(anova_results)
    print(anova_df.to_string(index=False))

    # Save results
    anova_df.to_json(stats_dir / 'anova_by_sector.json', orient='records', indent=2)
    print(f'  ✓ Saved: anova_by_sector.json')

# ============================================================================
# 9.2 Comprehensive Hypothesis Testing (eda_utils Integration)
# ============================================================================
if test_metrics and 'sector' in df.columns:
    print('\n🧪 Running Comprehensive Hypothesis Tests...')
    hypothesis_results = eda.perform_comprehensive_hypothesis_tests(
        df, metrics=test_metrics, group_column='sector', alpha=0.05
    )

    # Display results
    if 'test_results' in hypothesis_results:
        results_df = pd.DataFrame(hypothesis_results['test_results'])
        print(results_df[['metric', 'test_name', 'p_value', 'is_significant']])

        # Save detailed results
        results_df.to_json(stats_dir / 'comprehensive_hypothesis_tests.json', orient='records', indent=2)
        print(f'  ✓ Saved: comprehensive_hypothesis_tests.json')

    # Hypothesis Test Heatmap (using eda utility results)
    print('\n🔥 Creating Hypothesis Test Heatmap...')
    if 'test_results' in hypothesis_results:
        # Create visualization from results
        viz_metrics = [r['metric'] for r in hypothesis_results['test_results']]
        viz_pvals = [r['p_value'] for r in hypothesis_results['test_results']]

        fig_hyp = go.Figure(data=go.Heatmap(
            z=[viz_pvals],
            x=viz_metrics, y=['Hypothesis Test'],
            colorscale='RdYlGn_r', zmin=0, zmax=0.1,
            text=[[f"p={p:.4f}" for p in viz_pvals]],
            texttemplate='%{text}', textfont={'size': 12},
        ))
        fig_hyp.update_layout(
            title='<b>Hypothesis Test Results: Sector Differences</b><br><sup>Green = Significant Difference</sup>',
            template=PLOTLY_TEMPLATE, height=300,
        )
        fig_hyp.write_html(stats_dir / 'hypothesis_test_heatmap_enhanced.html')
        fig_hyp.show()
        print(f'  ✓ Saved: hypothesis_test_heatmap_enhanced.html')

print('\n✓ Hypothesis Testing Complete')


📊 HYPOTHESIS TESTING & STATISTICAL BENCHMARKING

📈 Running ANOVA Tests by Sector...
                 Metric  F_Statistic  P_Value  Significant
                    roe       0.7505 0.646768        False
      price_momentum_1m      18.2533 0.000000         True
         debt_to_equity       1.4379 0.175015        False
 earnings_quality_score       9.0865 0.000000         True
surprise_momentum_score       2.1289 0.029927         True
  ✓ Saved: anova_by_sector.json

🧪 Running Comprehensive Hypothesis Tests...

🔥 Creating Hypothesis Test Heatmap...

✓ Hypothesis Testing Complete


## Cell 9.5: Peer Analysis & Metric Trends


In [17]:
# ============================================================================
# Cell 9.5: Peer Analysis & Metric Trends
# ============================================================================

print('=' * 80)
print('🔍 PEER ANALYSIS & METRIC TRENDS')
print('=' * 80)

peer_dir = OUTPUT_DIR / 'eda' / 'benchmarking'
peer_dir.mkdir(parents=True, exist_ok=True)

# Select a few representative tickers for demonstration
if not df.empty:
    # Pick top 3 by market cap
    sample_tickers = df.sort_values('market_cap', ascending=False)['ticker'].head(3).tolist()

    for ticker in sample_tickers:
        print(f'\n👥 Analyzing Peer Group for {ticker}...')

        # 1. Find Peer Group
        peers = eda.find_peer_group(df, ticker=ticker, n_peers=5, criteria='market_cap')
        if not peers.empty:
            print(f"  Found {len(peers)} peers in same sector.")
            print(f"  Peers: {', '.join(peers['ticker'].tolist())}")

        # 2. Compare to Peers
        metrics_to_compare = ['p_e_ratio', 'roe', 'revenue_growth_yoy', 'debt_to_equity']
        metrics_to_compare = [m for m in metrics_to_compare if m in df.columns]

        if metrics_to_compare:
            comparison_results = eda.compare_to_peers(df, ticker=ticker, metrics=metrics_to_compare)
            if comparison_results:
                # compare_to_peers returns a dict, not a Figure. Let's print it or summarize it.
                print(f"    Peer comparison for {ticker} complete.")
                for m, stats in comparison_results.items():
                    print(
                        f"      {m}: Target={stats['target']:.2f}, Peer Avg={stats['peers_mean']:.2f}, Dev={stats['deviation_pct']:.1f}%")

        # 3. Analyze Metric Trend (if date-like columns exist)
        # Note: This usually needs time-series data, but we can demonstrate with available fiscal data
        if 'last_price' in df.columns:
            # Demonstration of trend analysis utility
            # (In a real scenario, this would use a historical dataframe)
            pass

# 4. Generate Benchmarking Report
print('\n📋 Generating Global Benchmarking Report...')
bench_metrics = ['roe', 'p_e_ratio', 'dividend_yield', 'market_cap']
bench_metrics = [m for m in bench_metrics if m in df.columns]

if bench_metrics:
    bench_report = eda.generate_benchmarking_report(
        df, metrics=bench_metrics, include_statistical_tests=True
    )
    # Save report summary
    with open(peer_dir / 'benchmarking_report_summary.json', 'w') as f:
        # Convert non-serializable parts if any
        json.dump({k: str(v) for k, v in bench_report.items() if k != 'figures'}, f, indent=2)
    print(f'  ✓ Saved: benchmarking_report_summary.json')

print('\n✓ Peer Analysis Complete')


🔍 PEER ANALYSIS & METRIC TRENDS

👥 Analyzing Peer Group for NVDA...
  Found 5 peers in same sector.
  Peers: AAPL, MSFT, AVGO, 2330, A005930
    Peer comparison for NVDA complete.
      p_e_ratio: Target=45.88, Peer Avg=8170.85, Dev=-99.4%
      roe: Target=83.43, Peer Avg=49.67, Dev=68.0%
      revenue_growth_yoy: Target=207.18, Peer Avg=26.37, Dev=685.8%
      debt_to_equity: Target=0.09, Peer Avg=0.58, Dev=-84.4%

👥 Analyzing Peer Group for GOOGL...
  Found 5 peers in same sector.
  Peers: META, 700, NFLX, 941, TMUS
    Peer comparison for GOOGL complete.
      p_e_ratio: Target=33.13, Peer Avg=60.90, Dev=-45.6%
      roe: Target=32.12, Peer Avg=23.49, Dev=36.7%
      revenue_growth_yoy: Target=25.40, Peer Avg=20.19, Dev=25.8%
      debt_to_equity: Target=0.11, Peer Avg=0.66, Dev=-82.8%

👥 Analyzing Peer Group for AAPL...
  Found 5 peers in same sector.
  Peers: MSFT, NVDA, AVGO, 2330, A005930
    Peer comparison for AAPL complete.
      p_e_ratio: Target=34.95, Peer Avg=8173.04, De

## Cell 10: Dashboard Summary & Artifact Index


In [18]:
# ============================================================================
# Cell 10: Dashboard Summary & Artifact Index
# ============================================================================

print('=' * 80)
print('📋 DASHBOARD SUMMARY & ARTIFACT INDEX')
print('=' * 80)

# Collect all generated artifacts
artifact_index = {
    'generated_at': datetime.now().isoformat(),
    'reference_date': REFERENCE_DATE.isoformat(),
    'data_shape': {'rows': df.shape[0], 'columns': df.shape[1]},
    'phase93_coverage': coverage_stats,
    'v115_features': {
        'price_target_dynamics': sum(1 for f in ['pt_momentum_1m', 'pt_acceleration_short', 
                                                   'pt_consensus_convergence'] if f in df.columns),
        'eps_trajectory': sum(1 for f in ['eps_cagr_5y', 'eps_quarterly_trend', 
                                           'eps_positive_streak'] if f in df.columns),
        'cashflow_temporal': sum(1 for f in ['fcf_quarterly_trend', 'cfo_margin_trend', 
                                              'investment_intensity_trend'] if f in df.columns),
        'valuation_timeseries': sum(1 for f in ['ev_sales_trend_1y', 'p_e_momentum_yoy',
                                                 'p_b_mean_reversion_signal'] if f in df.columns),
        'dividend_reliability': sum(1 for f in ['dividend_yield_volatility', 'dividend_yield_trend',
                                                 'dividend_payout_growth'] if f in df.columns),
    },
    'artifacts': {},
}

# Scan output directories (including new v1.15 directories)
for subdir in ['dashboards', 'by_region', 'by_sector', 'by_industry', 'by_exchange',
               'earnings_analytics', 'dividend_visualizations', 'employment_analytics',
               'advanced_analytics', 'earnings_visualizations', 'visualizations',
               'price_target_dynamics', 'eps_trajectory_analytics', 
               'cashflow_temporal_analytics', 'temporal_analytics']:
    dir_path = OUTPUT_DIR / 'eda' / subdir
    if dir_path.exists():
        files = list(dir_path.glob('*.html')) + list(dir_path.glob('*.json'))
        artifact_index['artifacts'][subdir] = [f.name for f in files]

# Save artifact index
with open(OUTPUT_DIR / 'eda' / 'artifact_index.json', 'w') as f:
    json.dump(artifact_index, f, indent=2, default=str)

# Print summary
print('\n📁 Generated Artifacts:')
for category, files in artifact_index['artifacts'].items():
    if files:
        print(f'\n  {category}/:')
        for f in files[:5]:
            print(f'    • {f}')
        if len(files) > 5:
            print(f'    ... and {len(files) - 5} more')

total_artifacts = sum(len(f) for f in artifact_index['artifacts'].values())
print(f'\n✓ Total Artifacts Generated: {total_artifacts}')
print(f'✓ Artifact Index: outputs/eda/artifact_index.json')

# Phase 9.3 Coverage Summary
print('\n' + '=' * 80)
print('📊 PHASE 9.3 FEATURE COVERAGE SUMMARY')
print('=' * 80)

print(f'\nCategories: {len(PHASE93_FEATURE_CATEGORIES)}')
print(f'Total Features Registered: {sum(len(f) for f in PHASE93_FEATURE_CATEGORIES.values())}')
print(f'Features Present in Data: {total_phase93}')
print(f'Overall Coverage: {total_phase93 / sum(len(f) for f in PHASE93_FEATURE_CATEGORIES.values()) * 100:.1f}%')

# v1.15 Feature Coverage Summary
v115_summary = artifact_index['v115_features']
print('\n' + '=' * 80)
print('📊 PHASE 9.3 v1.15 FEATURE ENHANCEMENT SUMMARY')
print('=' * 80)

print(f'\n🆕 New v1.15 Feature Generators:')
print(f'   Price Target Dynamics:  {v115_summary["price_target_dynamics"]}/35 features')
print(f'   EPS Trajectory:         {v115_summary["eps_trajectory"]}/14 features')
print(f'   Cash Flow Temporal:     {v115_summary["cashflow_temporal"]}/12 features')
print(f'   Valuation Timeseries:   {v115_summary["valuation_timeseries"]}/22 features')
print(f'   Dividend Reliability:   {v115_summary["dividend_reliability"]}/26 features')

print('\n✓ Stock Analytics Dashboard Complete!')


📋 DASHBOARD SUMMARY & ARTIFACT INDEX

📁 Generated Artifacts:

  dashboards/:
    • exchange_sector_treemap.html
    • industry_distribution.html
    • region_exchange_heatmap.html
    • region_sector_sunburst.html

  by_region/:
    • regional_boxplots.html
    • regional_radar_comparison.html
    • regional_valuation_comparison.html
    • regional_statistics.json

  by_sector/:
    • sector_category_coverage.html
    • sector_distribution_comparison.html
    • sector_performance_heatmap.html
    • sector_valuation_scatter.html

  earnings_analytics/:
    • analyst_consensus_dashboard.html
    • analyst_recommendations.html
    • analyst_recommendation_heatmap.html
    • category_comparison_chart.html
    • earnings_calendar.html
    ... and 16 more

  dividend_visualizations/:
    • dividend_reliability_dashboard.html
    • dividend_scorecard.html

  employment_analytics/:
    • efficiency_by_sector.html
    • employee_productivity.html
    • employee_productivity_dashboard.html

  ad

## Cell 11: Feature Registry Validation (v1.15)

Validate that all registered feature generators from FEATURE_REGISTRY
have produced their expected features in the DataFrame.


In [19]:
# ============================================================================
# Cell 11: Feature Registry Validation (Phase 9.3 v1.15)
# ============================================================================

print('=' * 80)
print('FEATURE REGISTRY VALIDATION (v1.15)')
print('=' * 80)

from finance_ml.features.advanced import FEATURE_REGISTRY, get_total_feature_count

registry_report = []
for key, entry in FEATURE_REGISTRY.items():
    category = entry['category']
    expected_count = entry['feature_count']
    
    # Get features from PHASE93_FEATURE_CATEGORIES
    if category in PHASE93_FEATURE_CATEGORIES:
        category_features = PHASE93_FEATURE_CATEGORIES[category]
        present = [f for f in category_features if f in df.columns]
        coverage = len(present) / len(category_features) * 100 if category_features else 0
    else:
        present = []
        coverage = 0
    
    registry_report.append({
        'Generator': key,
        'Category': category,
        'Expected': expected_count,
        'Present': len(present),
        'Coverage %': round(coverage, 1)
    })

registry_df = pd.DataFrame(registry_report)
print(registry_df.to_string(index=False))

total_expected = get_total_feature_count()
total_present = sum(registry_df['Present'])
print(f'\n✓ Total Feature Coverage: {total_present}/{total_expected} ({total_present/total_expected*100:.1f}%)')

print('\n✓ Feature Registry Validation Complete!')


FEATURE REGISTRY VALIDATION (v1.15)
            Generator               Category  Expected  Present  Coverage %
            valuation       Valuation Ratios        10       24        96.0
 valuation_timeseries   Valuation Timeseries        22       16       100.0
        profitability          Profitability        17       17       100.0
        margin_trends          Profitability         6       17       100.0
             momentum   Momentum & Technical        23       25       100.0
   technical_analysis     Technical Analysis        18       13        86.7
market_microstructure       Market Sentiment         5        3        75.0
   accounting_quality         Quality & Risk        18       20       100.0
   financial_distress         Quality & Risk         3       20       100.0
    cash_flow_quality              Cash Flow         5       17       100.0
    cashflow_temporal              Cash Flow        12       17       100.0
   capital_allocation     Capital Allocation        